# Finding the most similar product

Finding the most similar product
Being able to compute similarity between embeddings is a key step
within embeddings applications.
In this exercise, you'll return to the products list of dictionaries
that you worked with previously,
which contains the embedded short descriptions you also created earlier.

You'll compare a piece of text to these embedded descriptions
to identify the most similar description.

In [1]:
import numpy as np
from DATA import products
from openai_helper import client, create_embeddings
from scipy.spatial import distance

# Extract a list of product short description form products
product_descriptions = [product["short_description"] for product in products]

# Create embeddings for each product description
response = client.embeddings.create(
    model="text-embedding-3-small", input=product_descriptions
)
response_dict = response.model_dump()

# Extract the embeddings from response_dict and store in products
for i, product in enumerate(products):
    product["embedding"] = response_dict["data"][i]["embedding"]

# Embed the search text
search_text = "soap"
search_embedding = create_embeddings(search_text)[0]

distances = []
for product in products:
    # Compute the consine distance for each product description
    dist = distance.cosine(search_embedding, product["embedding"])
    distances.append(dist)

# Find and print the most similar product short_description
min_dist_ind = np.argmin(distances)
print(products[min_dist_ind]["short_description"])

Pamper yourself or a loved one with this spa gift basket full of relaxation goodies.


# Semantic search for products

Time to put your find_n_closest() function to use!
You'll test out your semantic product search on a test query,
computing a sorted list of the five most semantically similar products,
based on the enriched data you gave the model.

In [2]:
from all_functions import find_n_closest
from DATA import products
from openai_helper import create_embeddings


# Define a function to combine the relerant features into a single string
def create_product_text(product):
    return f"""
Title: {product["title"]}
Description: {product["short_description"]}
Category: {product["category"]}
Features: {", ".join(product["features"])}
"""


# Combine the features for each product
product_texts = [create_product_text(product) for product in products]

# Create the embeddings from product_texts
product_embeddings = create_embeddings(product_texts)

# Create the query vector from query_text
query_text = "computer"
query_vector = create_embeddings(query_text)[0]

# Find the five closest distances
hits = find_n_closest(query_vector, product_embeddings, n=5)

print(f"Search results for '{query_text}'")
for hit in hits:
    # Extract the product at each index in hits
    product = products[hit["index"]]
    print(product["title"])

Search results for 'computer'
High-Performance Gaming Laptop
Robot Building Kit
Smartphone X1
Smart Thermostat
Interactive Robot Pet


# Product recommendation system

In this exercise, you'll make a recommendation system for an online retailer
that sells a variety of products.
This system recommends three similar products to users
who visit a product page but don't purchase, based on the last product they visited.

In [5]:
from all_functions import create_product_text, find_n_closest
from DATA import products
from openai_helper import create_embeddings

last_product = {
    "title": "Building Blocks Deluxe Set",
    "short_description": "Unleash your creativity with this deluxe set of building blocks for endless fun.",
    "price": 34.99,
    "category": "Toys",
    "features": [
        "Includes 500+ colorful building blocks",
        "Promotes STEM learning and creativity",
        "Compatible with other major brick brands",
        "Comes with a durable storage container",
        "Ideal for children ages 3 and up",
    ],
}

# Combine the features for last_product and each product in products
last_product_text = create_product_text(last_product)
product_texts = [create_product_text(product) for product in products]

# Embed last_product_text and product_texts
last_product_embeddings = create_embeddings(last_product_text)[0]
product_embeddings = create_embeddings(product_texts)

# Fine the three smallest cosine distances and their indexes
hits = find_n_closest(last_product_embeddings, product_embeddings)

for hit in hits:
    product = products[hit["index"]]
    print(product["title"])

Robot Building Kit
LEGO Space Shuttle
Designer Makeup Brush Set
